# Bronze: source system extracts

Lands **synthetic** extracts shaped like the two systems a housing provider typically
runs: a property management system (units, tenancies, rent charges, occupancy) and a
finance system (receipts). Written verbatim, exactly as an extract would arrive.

| | |
| --- | --- |
| **Writes** | six `bronze_*` tables |
| **Grain** | one row per source record, no cleaning, no conforming |

> **No real data.** Every building, unit, household and payment here is generated.
> No TCHC system was contacted. The shapes and vocabulary are realistic for Ontario
> social housing — RGI and market rent, Section 3 arrears, unit turnover — so the
> modelling decisions are the ones you would actually face.

### Why the data is deliberately untidy

Bronze mirrors what real extracts look like: two systems that disagree on casing,
dates arriving as strings in different formats, money as text with currency symbols,
a duplicate batch, and a handful of orphan keys. If bronze were clean, Silver would
have nothing to demonstrate and the data-quality conversation would be theoretical.

In [ ]:
BUILDING_COUNT = 120
MONTHS_OF_HISTORY = 24
RANDOM_SEED = 20260827
PIPELINE_RUN_ID = ""

In [ ]:
import json
import random
import uuid
from datetime import date, datetime, timedelta, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    IntegerType, StringType, StructField, StructType,
)

RUN_ID = PIPELINE_RUN_ID or str(uuid.uuid4())
EXTRACTED_AT = datetime.now(timezone.utc).isoformat()
rng = random.Random(RANDOM_SEED)

# Anchor the calendar so every run of the demo produces the same story.
PERIOD_END = date(2026, 7, 31)
FIRST_PERIOD = date(PERIOD_END.year, PERIOD_END.month, 1) - timedelta(days=31 * (MONTHS_OF_HISTORY - 1))
FIRST_PERIOD = date(FIRST_PERIOD.year, FIRST_PERIOD.month, 1)

print(json.dumps({"run_id": RUN_ID, "buildings": BUILDING_COUNT,
                  "months": MONTHS_OF_HISTORY,
                  "first_period": FIRST_PERIOD.isoformat(),
                  "period_end": PERIOD_END.isoformat()}, indent=1))

In [ ]:
def month_starts(first, count):
    periods, year, month = [], first.year, first.month
    for _ in range(count):
        periods.append(date(year, month, 1))
        month += 1
        if month > 12:
            month, year = 1, year + 1
    return periods


PERIODS = month_starts(FIRST_PERIOD, MONTHS_OF_HISTORY)

WARDS = ["Etobicoke North", "Etobicoke Centre", "York South", "Humber River",
         "Eglinton-Lawrence", "Don Valley West", "Don Valley North", "Scarborough North",
         "Scarborough Centre", "Scarborough Southwest", "Toronto Centre",
         "Toronto-Danforth", "Davenport", "Parkdale-High Park", "Willowdale"]
REGIONS = {"Etobicoke North": "West", "Etobicoke Centre": "West", "York South": "West",
           "Humber River": "West", "Parkdale-High Park": "West", "Davenport": "West",
           "Eglinton-Lawrence": "North", "Don Valley West": "North",
           "Don Valley North": "North", "Willowdale": "North",
           "Scarborough North": "East", "Scarborough Centre": "East",
           "Scarborough Southwest": "East", "Toronto Centre": "Central",
           "Toronto-Danforth": "Central"}
PROPERTY_TYPES = ["Apartment tower", "Walk-up", "Townhouse", "Scattered house",
                  "Seniors designated"]
STREETS = ["Lawrence", "Kipling", "Jane", "Finch", "Sheppard", "Danforth", "Dundas",
           "Queen", "Bathurst", "Weston", "Markham", "Victoria Park", "Islington",
           "Eglinton", "Steeles", "Warden", "Birchmount", "Dufferin"]

## 1. Property management system: buildings and units

In [ ]:
buildings, units = [], []
unit_seq = 100000

for index in range(1, BUILDING_COUNT + 1):
    ward = rng.choice(WARDS)
    property_type = rng.choices(PROPERTY_TYPES, weights=[38, 25, 20, 9, 8])[0]
    if property_type == "Apartment tower":
        size = rng.randint(90, 260)
    elif property_type == "Walk-up":
        size = rng.randint(24, 80)
    elif property_type == "Townhouse":
        size = rng.randint(12, 48)
    elif property_type == "Scattered house":
        size = rng.randint(1, 4)
    else:
        size = rng.randint(60, 180)

    building_id = f"BLD{index:04d}"
    buildings.append({
        "building_id": building_id,
        # Source casing is inconsistent between records -- Silver has to normalise it.
        "building_name": (f"{rng.randint(20, 3400)} {rng.choice(STREETS)} "
                          f"{rng.choice(['Ave', 'St', 'Rd', 'Blvd', 'Cres'])}"),
        "ward_name": ward if index % 7 else ward.upper(),
        "region": REGIONS[ward],
        "property_type": property_type,
        "year_built": str(rng.randint(1958, 2012)),
        "total_units": size,
        "extract_batch": "PMS_BUILDING_FULL",
        "extracted_at": EXTRACTED_AT,
        "run_id": RUN_ID,
    })

    for number in range(1, size + 1):
        unit_seq += 1
        bedrooms = rng.choices([0, 1, 2, 3, 4], weights=[8, 38, 32, 18, 4])[0]
        units.append({
            "unit_id": f"U{unit_seq}",
            "building_id": building_id,
            "unit_number": f"{rng.randint(1, 24)}{number:02d}",
            "bedroom_count": bedrooms,
            # Free text in the source; Silver maps it to a controlled vocabulary.
            "unit_type": rng.choice(["RGI", "rgi", "Rent-Geared-to-Income",
                                     "Market", "MARKET"]),
            "accessible_flag": rng.choices(["Y", "N", None], weights=[8, 88, 4])[0],
            "extract_batch": "PMS_UNIT_FULL",
            "extracted_at": EXTRACTED_AT,
            "run_id": RUN_ID,
        })

print(f"buildings: {len(buildings):,}   units: {len(units):,}")

## 2. Tenancies, monthly rent charges, and occupancy

Arrears is a balance, not an event: it is the accumulation of charges not covered by
receipts. Generating charges and payments separately -- rather than an arrears figure
directly -- is what makes the Gold layer a real calculation rather than a passthrough.

In [ ]:
INCOME_BANDS = ["Under $15k", "$15k-$25k", "$25k-$35k", "$35k-$50k", "Over $50k"]

households, charges, payments, occupancy = [], [], [], []
household_seq = 500000

for unit in units:
    # A minority of units turn over during the window; the rest hold one tenancy.
    turnovers = rng.choices([0, 1, 2], weights=[78, 19, 3])[0]
    cursor = FIRST_PERIOD
    for spell in range(turnovers + 1):
        household_seq += 1
        household_id = f"H{household_seq}"
        is_rgi = unit["unit_type"].strip().lower() in ("rgi", "rent-geared-to-income")

        move_in = cursor if spell == 0 else cursor + timedelta(days=rng.randint(20, 75))
        if move_in > PERIOD_END:
            break
        remaining = [p for p in PERIODS if p >= date(move_in.year, move_in.month, 1)]
        spell_length = (len(remaining) if spell == turnovers
                        else rng.randint(4, max(5, len(remaining) - 3)))
        spell_periods = remaining[:spell_length]
        if not spell_periods:
            break

        # RGI rent tracks income; market rent tracks unit size.
        if is_rgi:
            band = rng.choices(INCOME_BANDS, weights=[26, 30, 22, 16, 6])[0]
            base_rent = {"Under $15k": 285, "$15k-$25k": 470, "$25k-$35k": 690,
                         "$35k-$50k": 940, "Over $50k": 1180}[band]
        else:
            band = rng.choices(INCOME_BANDS, weights=[4, 10, 22, 34, 30])[0]
            base_rent = 980 + unit["bedroom_count"] * 265
        base_rent += rng.randint(-40, 40)

        households.append({
            "household_ref": household_id,
            "unit_id": unit["unit_id"],
            "household_size": max(1, unit["bedroom_count"] + rng.randint(-1, 2)),
            "income_band": band,
            "subsidy_type": "RGI" if is_rgi else "Market",
            "move_in_dt": move_in.strftime("%d/%m/%Y"),   # dd/mm/yyyy in this system
            "move_out_dt": (None if spell == turnovers else
                            (spell_periods[-1] + timedelta(days=rng.randint(2, 26)))
                            .strftime("%d/%m/%Y")),
            "extract_batch": "PMS_TENANCY_FULL",
            "extracted_at": EXTRACTED_AT,
            "run_id": RUN_ID,
        })

        # A minority of households fall behind, and a few fall a long way behind.
        arrears_profile = rng.choices(["current", "occasional", "persistent", "severe"],
                                      weights=[70, 17, 9, 4])[0]
        pay_rate = {"current": 1.0, "occasional": 0.93,
                    "persistent": 0.78, "severe": 0.52}[arrears_profile]

        for period in spell_periods:
            charges.append({
                "charge_id": f"C{len(charges) + 1:08d}",
                "household_ref": household_id,
                "unit_id": unit["unit_id"],
                "charge_period": period.strftime("%Y-%m"),
                "charge_type": "RENT",
                "charge_amount": f"${base_rent:,.2f}",   # money as text with a symbol
                "extract_batch": "PMS_CHARGE_MONTHLY",
                "extracted_at": EXTRACTED_AT,
                "run_id": RUN_ID,
            })

            # Receipts come from the finance system: different key, different date
            # format, occasionally split across two payments, sometimes late.
            if rng.random() < pay_rate:
                paid = base_rent
            elif rng.random() < 0.45:
                paid = round(base_rent * rng.uniform(0.3, 0.85), 2)
            else:
                paid = 0.0
            if paid <= 0:
                continue
            splits = [paid] if rng.random() > 0.12 else [
                round(paid * 0.6, 2), round(paid * 0.4, 2)]
            for part in splits:
                offset = rng.choices([rng.randint(0, 5), rng.randint(6, 20),
                                      rng.randint(21, 55)], weights=[68, 24, 8])[0]
                paid_on = period + timedelta(days=offset)
                if paid_on > PERIOD_END:
                    continue
                payments.append({
                    "receipt_id": f"R{len(payments) + 1:08d}",
                    "tenant_account": household_id,
                    "posting_date": paid_on.strftime("%m/%d/%Y"),  # mm/dd/yyyy here
                    "amount_cad": f"{part:.2f}",
                    "payment_method": rng.choices(
                        ["PAD", "Cheque", "Online", "In person", "Money order"],
                        weights=[46, 14, 28, 8, 4])[0],
                    "source_system": "FIN",
                    "extract_batch": "FIN_RECEIPTS",
                    "extracted_at": EXTRACTED_AT,
                    "run_id": RUN_ID,
                })

        # Occupancy spell, and the vacancy that follows a move-out.
        occupancy.append({
            "event_id": f"O{len(occupancy) + 1:08d}",
            "unit_id": unit["unit_id"],
            "household_ref": household_id,
            "occupied_from": spell_periods[0].isoformat(),
            "occupied_to": (None if spell == turnovers
                            else (spell_periods[-1] + timedelta(days=rng.randint(2, 26)))
                            .isoformat()),
            "vacate_reason": (None if spell == turnovers else
                              rng.choice(["Transfer", "Moved out", "Eviction",
                                          "Deceased", "Abandoned"])),
            "extract_batch": "PMS_OCCUPANCY",
            "extracted_at": EXTRACTED_AT,
            "run_id": RUN_ID,
        })
        if spell != turnovers:
            cursor = spell_periods[-1] + timedelta(days=rng.randint(20, 90))
            if cursor > PERIOD_END:
                break

print(f"households: {len(households):,}  charges: {len(charges):,}  "
      f"payments: {len(payments):,}  occupancy: {len(occupancy):,}")

## 3. Introduce the defects a real extract carries

Each of these is something Silver must handle explicitly, and each is a talking point:
a re-sent batch, receipts for accounts that are not in the tenancy extract, and a work
order feed whose unit references do not all resolve.

In [ ]:
# A duplicated batch -- the classic "the extract ran twice" incident.
duplicate_slice = payments[:max(1, len(payments) // 120)]
payments.extend([dict(row) for row in duplicate_slice])

# Receipts whose account never appears in the tenancy extract.
for index in range(40):
    payments.append({
        "receipt_id": f"R9{index:07d}",
        "tenant_account": f"H{rng.randint(900000, 999999)}",
        "posting_date": (PERIOD_END - timedelta(days=rng.randint(1, 120))).strftime("%m/%d/%Y"),
        "amount_cad": f"{rng.uniform(50, 900):.2f}",
        "payment_method": "Cheque",
        "source_system": "FIN",
        "extract_batch": "FIN_RECEIPTS",
        "extracted_at": EXTRACTED_AT,
        "run_id": RUN_ID,
    })

# Unit turnaround work orders, with a few unresolvable unit references.
work_orders = []
for event in occupancy:
    if not event["occupied_to"]:
        continue
    vacated = date.fromisoformat(event["occupied_to"])
    ready = vacated + timedelta(days=rng.choices(
        [rng.randint(4, 20), rng.randint(21, 60), rng.randint(61, 180)],
        weights=[54, 34, 12])[0])
    work_orders.append({
        "work_order_id": f"W{len(work_orders) + 1:07d}",
        "unit_ref": event["unit_id"],
        "vacated_date": vacated.isoformat(),
        "ready_to_rent_date": ready.isoformat() if ready <= PERIOD_END else None,
        "turnaround_category": rng.choices(
            ["Standard clean", "Minor repair", "Major repair", "Capital"],
            weights=[46, 32, 17, 5])[0],
        "extract_batch": "PMS_WORKORDER",
        "extracted_at": EXTRACTED_AT,
        "run_id": RUN_ID,
    })
for index in range(25):
    work_orders.append({
        "work_order_id": f"W9{index:06d}",
        "unit_ref": f"U{rng.randint(1, 99999)}",
        "vacated_date": (PERIOD_END - timedelta(days=rng.randint(30, 300))).isoformat(),
        "ready_to_rent_date": None,
        "turnaround_category": "Standard clean",
        "extract_batch": "PMS_WORKORDER",
        "extracted_at": EXTRACTED_AT,
        "run_id": RUN_ID,
    })

print(f"work orders: {len(work_orders):,}  "
      f"(payments now {len(payments):,} including a re-sent batch)")

In [ ]:
def write_bronze(rows, table_name):
    """Everything lands as string-typed text, the way a flat extract actually arrives."""
    if not rows:
        print(f"  {table_name}: no rows")
        return
    columns = sorted({key for row in rows for key in row})
    schema = StructType([StructField(column, StringType(), True) for column in columns])
    records = [tuple(None if row.get(column) is None else str(row.get(column))
                     for column in columns) for row in rows]
    frame = spark.createDataFrame(records, schema=schema)
    spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    try:
        (frame.write.format("delta").mode("overwrite")
            .partitionBy("run_id").saveAsTable(table_name))
    except Exception as error:
        print(f"  WARNING {table_name} schema changed - replacing all runs "
              f"({str(error).splitlines()[0][:100]})")
        spark.conf.set("spark.sql.sources.partitionOverwriteMode", "static")
        try:
            (frame.write.format("delta").mode("overwrite").partitionBy("run_id")
                .option("overwriteSchema", "true").saveAsTable(table_name))
        finally:
            spark.conf.set("spark.sql.sources.partitionOverwriteMode", "dynamic")
    print(f"  {table_name:28} {frame.count():>8,} rows")


print("writing bronze:")
write_bronze(buildings, "bronze_pms_buildings")
write_bronze(units, "bronze_pms_units")
write_bronze(households, "bronze_pms_tenancies")
write_bronze(charges, "bronze_pms_rent_charges")
write_bronze(payments, "bronze_fin_receipts")
write_bronze(occupancy, "bronze_pms_occupancy")
write_bronze(work_orders, "bronze_pms_work_orders")

In [ ]:
display(spark.read.table("bronze_fin_receipts")
        .filter(F.col("run_id") == RUN_ID)
        .select("receipt_id", "tenant_account", "posting_date", "amount_cad",
                "payment_method")
        .limit(10))